# Visualización — Simulación de Torre de Secado por Aspersión

Este notebook ejecuta la simulación del proceso de secado por aspersión y genera todas las gráficas de resultados, comparando con datos experimentales.

Basado en el trabajo de Santiago González Gallego y Lina Steffania.

## 1. Configuración del entorno

In [ ]:
# Clonar el repositorio
!git clone https://github.com/Extantword/fede.git /content/fede 2>/dev/null || echo "Repositorio ya clonado"

# Cambiar al directorio del proyecto
import os
os.chdir('/content/fede')

# Instalar dependencias
!pip install -q numba

In [ ]:
import numpy as np
import math as mt
import matplotlib.pyplot as plt
import time

# Configuración de gráficas para el notebook
plt.rcParams["font.family"] = "serif"
plt.rcParams["figure.dpi"] = 120
%matplotlib inline

# Importar módulos del proyecto
from simulacion import Spray
from constantes import *
from datos_experimentales import *

print("Módulos importados correctamente.")

## 2. Ejecutar la simulación

La primera ejecución incluye el tiempo de compilación JIT de Numba (~30-60s extra).

In [ ]:
# Parámetros de simulación
T = 50000       # Número de tandas
h_eu = 0.0005   # Paso de tiempo de Euler [s]
Part = 80000    # Máx iteraciones por tanda

# Vectores iniciales
Mov_VarTemp_1 = np.zeros(T)
T_EpTemp_1    = np.zeros(T)
Y_p_1         = np.zeros(T)
M_wDL_1       = np.zeros(T)

print("Comenzando la simulación...")
start = time.process_time()

Vector = Spray(T, h_eu, Part, Mov_VarTemp_1, T_EpTemp_1, Y_p_1, M_wDL_1)

elapsed = time.process_time() - start
print(f"Simulación completada en {elapsed:.1f} s")

In [ ]:
# Extraer resultados usando los índices nombrados
i       = Vector[IDX_FINAL_I]
Mov_Z   = Vector[IDX_MOV_Z]
T_Eg    = Vector[IDX_T_EG]
Y_graf  = Vector[IDX_Y_GRAF]
T_R     = Vector[IDX_T_R]
Mov_R   = Vector[IDX_MOV_R][0:i]
D_graf  = Vector[IDX_D_DROP]
X_graf  = Vector[IDX_X_W]
T_Eout  = Vector[IDX_T_EOUT]
Y_Eout  = Vector[IDX_Y_EOUT]
T_ECT   = Vector[IDX_T_ECT]
Z_Mout  = Vector[IDX_Z_MOUT]
P_inMV  = Vector[IDX_P_INMV]
m_LMV   = Vector[IDX_M_LMV]
f_bal   = Vector[IDX_F_BALANC]

Mov_ZR = Mov_Z
Mov_ZE = Mov_Z

print(f"Última iteración de la gota: {i}")
print(f"Balance de masa (f_balanc): {f_bal:.4f} kg/h")
print(f"Puntos de datos temporales registrados: {len(T_Eout)}")

---
## 3. Gráficas de resultados

### Función auxiliar de estilo

In [ ]:
def estilo_ejes(ax):
    """Aplica el estilo estándar a un par de ejes."""
    ax.grid(True)
    ax.grid(color='#CCCCCC', linestyle='dotted', linewidth=0.5)
    for spine in ['top', 'bottom', 'left', 'right']:
        ax.spines[spine].set_linewidth(0.5)

### 3.1 Trayectoria de la gota (posición radial vs axial)

In [ ]:
fig, ax = plt.subplots(1, figsize=(5, 7))

ax.plot(Mov_R, Mov_Z, linewidth=0.5, color='k', label='Simulación')
ax.scatter(trayectoria_r, trayectoria_z, color='k', s=15, zorder=5, label='Experimental')

ax.set_xlabel("Radial distance from nozzle [m]")
ax.set_ylabel("Axial distance from nozzle [m]")
ax.set_title("Trayectoria de la gota")
estilo_ejes(ax)

plt.gca().invert_yaxis()
plt.yticks(np.arange(0, mt.ceil(Mov_Z[-1]), 0.25))
plt.xticks(np.arange(0, 0.7, 0.05))
ax.legend()

plt.tight_layout()
plt.show()

### 3.2 Perfil de temperatura vs distancia axial

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(Mov_ZR, T_R[0:i], linewidth=0.8, color='b', label='Fase refinada (gota)')
ax.plot(Mov_ZE, T_Eg[0:i], linewidth=0.8, color='r', label='Fase extracto (aire)')
ax.scatter(temp_axial_x, temp_axial_y, color='k', s=15, zorder=5, label='Experimental')

ax.set_xlabel("Axial distance from nozzle [m]")
ax.set_ylabel("Temperature [°C]")
ax.set_title("Perfil de temperatura a lo largo de la torre")
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.3 Humedad del aire vs distancia axial

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(Mov_ZE, Y_graf, linewidth=0.8, color='r', label='Simulación')
ax.scatter(humedad_axial_x, humedad_axial_y, color='k', s=15, zorder=5, label='Experimental')

ax.set_xlabel("Axial distance from nozzle [m]")
ax.set_ylabel("Extract humidity [kg/kg]")
ax.set_title("Humedad absoluta del aire vs posición axial")
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.4 Contenido de humedad del refinado vs distancia axial

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(Mov_ZE, X_graf, linewidth=0.8, color='r', label='Simulación')

ax.set_xlabel("Axial distance from nozzle [m]")
ax.set_ylabel("Refined moisture content [kg water / kg maltodextrin]")
ax.set_title("Contenido de humedad de la gota vs posición axial")
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.5 Diámetro de la gota vs distancia axial

In [ ]:
fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(Mov_ZE, D_graf, linewidth=0.8, color='r', label='Simulación')

ax.set_xlabel("Axial distance from nozzle [m]")
ax.set_ylabel("Droplet diameter [m]")
ax.set_title("Evolución del diámetro de la gota")
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.6 Temperatura de salida vs tiempo (con datos experimentales)

In [ ]:
# Ajuste de los primeros puntos como en el código original
T_Eout_plot = list(T_Eout)  # Copiar para no mutar el original
rango_limite = min(100, len(T_Eout_plot))
for u in range(rango_limite):
    T_Eout_plot[u] = 75.0

EjX = np.linspace(0, 25, len(T_Eout_plot))

fig, ax = plt.subplots(1, figsize=(12, 5))

ax.plot(EjX, T_Eout_plot, linewidth=0.8, color='b', label='Simulación')
ax.scatter(X_Exp2, Y_Exp2, color='k', s=8, zorder=5, label='Exp. (descenso)')
ax.scatter(X_Exp3, Y_Exp3, color='r', s=8, zorder=5, marker='^', label='Exp. (ascenso)')

ax.axhline(85, linewidth=0.5, color='black', ls='--', label='T = 85 °C')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Outlet temperature [°C]")
ax.set_title("Temperatura de salida del aire vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.yticks(np.arange(70, 90, 1))
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 500))
estilo_ejes(ax)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

### 3.7 Humedad del aire de salida vs tiempo

In [ ]:
EjX = np.linspace(0, 25, len(T_Eout))

fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(EjX, Y_Eout, linewidth=0.8, color='b', label='Simulación')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Humidity of outlet air [kg/kg]")
ax.set_title("Humedad del aire de salida vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.8 Temperatura de entrada vs tiempo (con datos experimentales)

In [ ]:
EjX = np.linspace(0, 25, len(T_Eout))

fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(EjX, T_ECT, linewidth=0.8, color='b', label='Simulación')
ax.scatter(Time_exp, T_exp, color='k', s=15, zorder=5, label='Experimental')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Inlet temperature [°C]")
ax.set_title("Temperatura de entrada del aire vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 100))
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.9 Contenido de humedad de salida vs tiempo

In [ ]:
EjX = np.linspace(0, 25, len(T_Eout))

fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(EjX, Z_Mout, linewidth=0.8, color='b', label='Simulación')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Moisture content [kg/kg]")
ax.set_title("Contenido de humedad del producto de salida vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.10 Presión de entrada del líquido vs tiempo

In [ ]:
EjX = np.linspace(0, 25, len(T_Eout))

fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(EjX, P_inMV, linewidth=0.8, color='b', label='Simulación')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Inlet pressure of the liquid [Pa]")
ax.set_title("Presión de entrada del líquido vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

### 3.11 Flujo del líquido vs tiempo

In [ ]:
EjX = np.linspace(0, 25, len(T_Eout))

fig, ax = plt.subplots(1, figsize=(10, 5))

ax.plot(EjX, m_LMV, linewidth=0.8, color='b', label='Simulación')
ax.axvline(x=80, linewidth=0.5, color='red', linestyle='-', alpha=0.5)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Liquid flow [kg/h]")
ax.set_title("Flujo del líquido vs tiempo")
ax.set_xlim([0, mt.ceil(EjX[-1])])
plt.xticks(np.arange(0, mt.ceil(EjX[-1]), 25))
estilo_ejes(ax)
ax.legend()

plt.tight_layout()
plt.show()

---
## 4. Verificación del balance de masa

In [ ]:
R_time = Vector[IDX_R_TIME]

print("=" * 50)
print("RESUMEN DE LA SIMULACIÓN")
print("=" * 50)
print(f"Balance de masa (f_balanc): {f_bal:.4f} kg/h")
print(f"Última iteración: {i}")

if len(R_time) > 0:
    promedio_rt = np.mean(np.array(R_time))
    print(f"Tiempo de residencia promedio: {promedio_rt:.4f} s")
else:
    print("No se registraron tiempos de residencia.")

if len(T_Eout) > 0:
    print(f"Temperatura de salida promedio: {np.mean(np.array(T_Eout)):.2f} °C")

if len(Y_Eout) > 0:
    print(f"Humedad de salida promedio: {np.mean(np.array(Y_Eout)):.6f} kg/kg")

print("=" * 50)